# Concrete Compressive Strength Prediction - Model Development

In [1]:
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.tree import export_text
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

RANDOM_SEED = 13

## 1 - Data preparation and preprocessing

- Use the exported merged data and add/modify the features.

In [2]:
DATA_PATH = Path().resolve() / "data"
dataset = DATA_PATH / "concrete_data_full.csv"
dataset

WindowsPath('D:/Users/Esperine/Workspace3/_projects/_dev/buhos/data/concrete_data_full.csv')

In [3]:
columns = [
    "sample_id", 
    "src", 
    "cement", 
    "blast_furnace_slag", 
    "fly_ash", 
    "silica_fume", 
    "coarse_aggregate", 
    "fine_aggregate", 
    "water", 
    "superplasticizer", 
    "age", 
    "compressive_strength"
]

columns_fe = [
    "log_age",
    "ratio_wb",
    "pct_scm_repl"
]

features = columns[2:-2] + columns_fe
target = "compressive_strength"

def setup_features(dataframe):
    df = dataframe.copy()

    df["log_age"] = np.log1p(df["age"])

    scm_mats = ["blast_furnace_slag", "fly_ash", "silica_fume"]
    total_scm = df[scm_mats].sum(axis=1)
    total_binder = df["cement"] + total_scm
    df["ratio_wb"] = df["water"].div(total_binder).replace([np.inf, -np.inf, np.nan], 0)
    df["pct_scm_repl"] = total_scm.div(total_binder).replace([np.inf, -np.inf, np.nan], 0)

    return df

In [4]:
# Helper functions for metrics

def get_r2_scores(y_true, y_pred, features):
    r2 = r2_score(y_true, y_pred)
    
    n_samples = len(y_true)
    n_features = len(features)
    adj_r2 = 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)

    return r2, adj_r2


def get_metrics(model, features_X, target_y):
    y_pred = model.predict(features_X)
    
    mae = mean_absolute_error(target_y, y_pred)
    rmse = root_mean_squared_error(target_y, y_pred)
    mape = mean_absolute_percentage_error(target_y, y_pred)
    r2, r2_adj = get_r2_scores(target_y, y_pred, list(features_X.columns))

    return mae, rmse, mape, r2, r2_adj


def predict(model, features_X, target_y, set_type):
    y_pred = model.predict(features_X)

    mae = mean_absolute_error(target_y, y_pred)
    rmse = root_mean_squared_error(target_y, y_pred)
    mape = mean_absolute_percentage_error(target_y, y_pred)
    r2, r2_adj = get_r2_scores(target_y, y_pred, list(features_X.columns))
    
    print(f"===== Scores - {set_type.upper()} =====")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAPE: {mape:.4f}")
    print(f"R2: {r2:.4f}")
    print(f"R2 Adjusted: {r2_adj:.4f}")
    
    return y_pred

In [5]:
features

['cement',
 'blast_furnace_slag',
 'fly_ash',
 'silica_fume',
 'coarse_aggregate',
 'fine_aggregate',
 'water',
 'superplasticizer',
 'log_age',
 'ratio_wb',
 'pct_scm_repl']

In [6]:
df_raw = pd.read_csv(dataset)
df = setup_features(df_raw)
df.head()

,sample_id,src,cement,blast_furnace_slag,fly_ash,silica_fume,coarse_aggregate,fine_aggregate,water,superplasticizer,age,compressive_strength,log_age,ratio_wb,pct_scm_repl
0,s1_1,src1,540.0,0.0,0.0,0.0,1040.0,676.0,162.0,2.5,28,79.99,3.367296,0.30000,0.0
1,s1_2,src1,540.0,0.0,0.0,0.0,1055.0,676.0,162.0,2.5,28,61.89,3.367296,0.30000,0.0
2,s1_3,src1,332.5,142.5,0.0,0.0,932.0,594.0,228.0,0.0,270,40.27,5.602119,0.48000,0.3
3,s1_4,src1,332.5,142.5,0.0,0.0,932.0,594.0,228.0,0.0,365,41.05,5.902633,0.48000,0.3
4,s1_5,src1,198.6,132.4,0.0,0.0,978.4,825.5,192.0,0.0,360,44.30,5.888878,0.58006,0.4


## 2 - Train-validation-test split

- Previously when we explored different models to establish baseline scores and measure feature importance, we only used a single stratified split.
- For searching and finetuning hyperparameters with cross-validation, we use stratified k-fold (or repeated stratified k-fold).
- When doing finetuning and cross-validation, use combined train and validation set.
- To automate the process, we use [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

In [7]:
# Stratify data based on quantiles of target (compressive strength)

df["strata_key"] = df["src"].astype(str) + "_" + pd.qcut(df[target], q=4, labels=["q1", "q2", "q3", "q4"]).astype(str)
stratify_key = df["strata_key"]
stratify_key

0       src1_q4
1       src1_q4
2       src1_q3
3       src1_q3
4       src1_q3
         ...   
1622    src2_q2
1623    src2_q4
1624    src2_q4
1625    src2_q4
1626    src2_q4
Name: strata_key, Length: 1627, dtype: str

In [8]:
# Setup features matrix and target vector

features_X = df[features]
target_y = df[target]

In [9]:
features_X.head()

,cement,blast_furnace_slag,fly_ash,silica_fume,coarse_aggregate,fine_aggregate,water,superplasticizer,log_age,ratio_wb,pct_scm_repl
0,540.0,0.0,0.0,0.0,1040.0,676.0,162.0,2.5,3.367296,0.30000,0.0
1,540.0,0.0,0.0,0.0,1055.0,676.0,162.0,2.5,3.367296,0.30000,0.0
2,332.5,142.5,0.0,0.0,932.0,594.0,228.0,0.0,5.602119,0.48000,0.3
3,332.5,142.5,0.0,0.0,932.0,594.0,228.0,0.0,5.902633,0.48000,0.3
4,198.6,132.4,0.0,0.0,978.4,825.5,192.0,0.0,5.888878,0.58006,0.4


In [10]:
target_y.head()

0    79.99
1    61.89
2    40.27
3    41.05
4    44.30
Name: compressive_strength, dtype: float64

In [11]:
X_full_train, X_test, y_full_train, y_test, key_full_train, key_test = train_test_split(
    features_X,
    target_y,
    stratify_key,
    test_size=0.2, 
    random_state=RANDOM_SEED, 
    stratify=stratify_key
)

X_train, X_val, y_train, y_val, key_train, key_val = train_test_split(
    X_full_train, 
    y_full_train,
    key_full_train,
    test_size=0.25, 
    random_state=RANDOM_SEED, 
    stratify=key_full_train
)

In [12]:
X_full_train.shape, X_test.shape, y_full_train.shape, y_test.shape, key_full_train.shape, key_test.shape

((1301, 11), (326, 11), (1301,), (326,), (1301,), (326,))

In [13]:
X_train.shape, X_val.shape, y_train.shape, y_val.shape, key_train.shape, key_val.shape

((975, 11), (326, 11), (975,), (326,), (975,), (326,))

In [14]:
# Setup stratified k-fold (using 10 fold split)

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
cv_full_train = list(skf.split(X_full_train, key_full_train))

## 3 - Model development (Random Forest)

Consider the following for finetuning.

- `n_estimators`
- `max_depth`
- `min_samples_leaf`

In [15]:
model = RandomForestRegressor(random_state=RANDOM_SEED)
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",13
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [16]:
# Baseline score for RF

y_pred_val = predict(
    model=model, 
    features_X=X_val, 
    target_y=y_val,
    set_type="validation"
)

===== Scores - VALIDATION =====
MAE: 4.4028
RMSE: 6.4561
MAPE: 0.1463
R2: 0.8761
R2 Adjusted: 0.8717


In [17]:
# Determine suitable number of estimators

model = RandomForestRegressor(random_state=RANDOM_SEED)

param_grid = {
    "n_estimators": [100, 200, 300, 400, 500]
}

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "mape": "neg_mean_absolute_percentage_error",
    "r2_score": "r2"
}

grid_search = GridSearchCV(
    model, 
    param_grid=param_grid, 
    cv=cv_full_train, 
    scoring=scoring,
    refit="mae",
    verbose=2
)

grid_search.fit(X_full_train, y_full_train)

Fitting 10 folds for each of 5 candidates, totalling 50 fits
[CV] END ...................................n_estimators=100; total time=   0.2s
[CV] END ...................................n_estimators=100; total time=   0.2s
[CV] END ...................................n_estimators=100; total time=   0.2s
[CV] END ...................................n_estimators=100; total time=   0.2s
[CV] END ...................................n_estimators=100; total time=   0.2s
[CV] END ...................................n_estimators=100; total time=   0.2s
[CV] END ...................................n_estimators=100; total time=   0.2s
[CV] END ...................................n_estimators=100; total time=   0.2s
[CV] END ...................................n_estimators=100; total time=   0.2s
[CV] END ...................................n_estimators=100; total time=   0.2s
[CV] END ...................................n_estimators=200; total time=   0.5s
[CV] END ...................................n_es

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=13)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'mae': 'neg_mean_absolute_error', 'mape': 'neg_mean_absolute_percentage_error', 'r2_score': 'r2', 'rmse': 'neg_root_mean_squared_error'}"
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'mae'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[(array([ 0, ...shape=(1170,)), ...), (array([ 0, ...shape=(1171,)), ...), ...]"
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are pri

In [18]:
result_cols = [
    "param_n_estimators",
    "mean_test_mae",
    "std_test_mae",
    "mean_test_rmse",
    "std_test_rmse",
    "mean_test_mape",
    "std_test_mape",
    "mean_test_r2_score",
    "std_test_r2_score",
    "rank_test_mae",
    "rank_test_rmse",
    "rank_test_mape",
    "rank_test_r2_score"
]

cv_results_df = pd.DataFrame(grid_search.cv_results_)
cv_results_df[result_cols][cv_results_df["rank_test_mae"] == 1]

,param_n_estimators,mean_test_mae,std_test_mae,mean_test_rmse,std_test_rmse,mean_test_mape,std_test_mape,mean_test_r2_score,std_test_r2_score,rank_test_mae,rank_test_rmse,rank_test_mape,rank_test_r2_score
2,300,-4.257998,0.282718,-6.38108,0.821336,-0.134479,0.01301,0.890612,0.029831,1,1,1,1


In [19]:
select_cols = [
    "mean_test_mae",
    "std_test_mae",
    "mean_test_rmse",
    "std_test_rmse",
    "mean_test_r2_score",
    "std_test_r2_score",
]

cv_results_df[result_cols].groupby("param_n_estimators")[select_cols].agg("mean")

,mean_test_mae,std_test_mae,mean_test_rmse,std_test_rmse,mean_test_r2_score,std_test_r2_score
param_n_estimators,,,,,,
100,-4.283390,0.282920,-6.414496,0.778514,0.889469,0.029307
200,-4.260641,0.281432,-6.383882,0.807003,0.890578,0.029400
300,-4.257998,0.282718,-6.381080,0.821336,0.890612,0.029831
400,-4.267922,0.281621,-6.393118,0.829284,0.890242,0.029943
500,-4.266335,0.283506,-6.384575,0.827334,0.890503,0.030024


Values are relatively close, consider `n_estimators` = 300

In [20]:
# Retrain to finetune using other parameters

model = RandomForestRegressor(random_state=RANDOM_SEED)

param_grid = {
    "n_estimators": [300],
    "max_depth": [1, 3, 5, 10, 15, 20, 25, None],
    "min_samples_leaf": [1, 2, 3, 5, 10]
}

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "mape": "neg_mean_absolute_percentage_error",
    "r2_score": "r2"
}

grid_search = GridSearchCV(
    model, 
    param_grid=param_grid, 
    cv=cv_full_train, 
    scoring=scoring,
    refit="mae",
    verbose=2
    # n_jobs=-1
)

grid_search.fit(X_full_train, y_full_train)

Fitting 10 folds for each of 40 candidates, totalling 400 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=13)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [1, 3, ...], 'min_samples_leaf': [1, 2, ...], 'n_estimators': [300]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'mae': 'neg_mean_absolute_error', 'mape': 'neg_mean_absolute_percentage_error', 'r2_score': 'r2', 'rmse': 'neg_root_mean_squared_error'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'mae'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here..

In [21]:
model_rfr = grid_search.best_estimator_
model_rfr

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",25
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",13
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of 

In [22]:
grid_search.best_params_

{'max_depth': 25, 'min_samples_leaf': 1, 'n_estimators': 300}

In [23]:
# This gives the best MAE
grid_search.best_score_

np.float64(-4.257764253897999)

In [24]:
cv_results_df = pd.DataFrame(grid_search.cv_results_)

result_cols = [
    "param_max_depth",
    "param_min_samples_leaf",
    "mean_test_mae",
    "std_test_mae",
    "mean_test_rmse",
    "std_test_rmse",
    "mean_test_mape",
    "std_test_mape",
    "mean_test_r2_score",
    "std_test_r2_score",
    "rank_test_mae",
    "rank_test_rmse",
    "rank_test_mape",
    "rank_test_r2_score"
]

cv_results_df[result_cols].iloc[grid_search.best_index_]

param_max_depth                 25
param_min_samples_leaf           1
mean_test_mae            -4.257764
std_test_mae              0.283071
mean_test_rmse           -6.380942
std_test_rmse             0.821743
mean_test_mape           -0.134472
std_test_mape             0.013009
mean_test_r2_score        0.890616
std_test_r2_score         0.029836
rank_test_mae                    1
rank_test_rmse                   1
rank_test_mape                   1
rank_test_r2_score               1
Name: 30, dtype: object

In [25]:
select_cols = [
    "mean_test_mae",
    "std_test_mae",
    "mean_test_rmse",
    "std_test_rmse",
    "mean_test_r2_score",
    "std_test_r2_score",
]

cv_results_df[result_cols].groupby("param_max_depth", as_index=False, dropna=False)[select_cols].agg("mean")

,param_max_depth,mean_test_mae,std_test_mae,mean_test_rmse,std_test_rmse,mean_test_r2_score,std_test_r2_score
0,1.0,-12.607863,0.904817,-15.844683,0.991962,0.339624,0.057034
1,3.0,-8.228617,0.392772,-10.999085,0.615960,0.680120,0.041086
2,5.0,-6.237522,0.296055,-8.547855,0.695496,0.805746,0.035768
3,10.0,-4.937984,0.279801,-7.111737,0.710718,0.864578,0.029110
4,15.0,-4.809549,0.278090,-7.003039,0.732919,0.868259,0.029779
5,20.0,-4.804682,0.277515,-6.999561,0.731852,0.868373,0.029775
6,25.0,-4.803602,0.277500,-6.998448,0.731806,0.868412,0.029771
7,NaN,-4.803651,0.277433,-6.998477,0.731727,0.868411,0.029770


In [26]:
mae = -cv_results_df["mean_test_mae"].iloc[grid_search.best_index_]
rmse = -cv_results_df["mean_test_rmse"].iloc[grid_search.best_index_]
mape = -cv_results_df["mean_test_mape"].iloc[grid_search.best_index_]
r2 = cv_results_df["mean_test_r2_score"].iloc[grid_search.best_index_]
n_samples, n_features = len(y_full_train), len(features)
r2_adj = 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)

results = [{
    "type": "random forest",
    "mae_val": mae,
    "rmse_val": rmse,
    "mape_val": mape,
    "r2_val": r2,
    "adj_r2_val": r2_adj
}]

scores_df = pd.DataFrame(results)
scores_df.round(4)

,type,mae_val,rmse_val,mape_val,r2_val,adj_r2_val
0,random forest,4.2578,6.3809,0.1345,0.8906,0.8897


## 4 - Model development (XGBoost)

Consider the following for finetuning.

- `eta` - the learning rate
- `max_depth` - the maximum depth of each tree
- `min_child_weight` - the minimal size of a leaf, similar to `min_samples_leaf` in decision tree or random forest.

In [27]:
model = xgb.XGBRegressor(random_state=RANDOM_SEED)
model.fit(X_train, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [28]:
# Baseline score for XGB

y_pred_val = predict(
    model=model, 
    features_X=X_val, 
    target_y=y_val,
    set_type="validation"
)

===== Scores - VALIDATION =====
MAE: 3.8733
RMSE: 5.6852
MAPE: 0.1258
R2: 0.9039
R2 Adjusted: 0.9005


For the number of estimators, set to 300 like in random forest for simplicity.

In [29]:
model = xgb.XGBRegressor(random_state=RANDOM_SEED)

param_grid = {
    "n_estimators": [300],
    "learning_rate": [0.01, 0.05, 0.1, 0.3],
    "max_depth": [3, 4, 6, 8, 10],
    "min_child_weight": [1, 2, 3, 5, 10]
}

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "mape": "neg_mean_absolute_percentage_error",
    "r2_score": "r2"
}

grid_search = GridSearchCV(
    model, 
    param_grid=param_grid, 
    cv=cv_full_train, 
    scoring=scoring,
    refit="mae",
    verbose=2
    # n_jobs=-1
)

grid_search.fit(X_full_train, y_full_train)

Fitting 10 folds for each of 100 candidates, totalling 1000 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 4, ...], 'min_child_weight': [1, 2, ...], 'n_estimators': [300]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'mae': 'neg_mean_absolute_error', 'mape': 'neg_mean_absolute_percentage_error', 'r2_score': 'r2', 'rmse': 'neg_root_mean_squared_error'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'mae'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validat

In [30]:
model_xgbr = grid_search.best_estimator_
model_xgbr

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [31]:
grid_search.best_params_

{'learning_rate': 0.3,
 'max_depth': 4,
 'min_child_weight': 2,
 'n_estimators': 300}

In [32]:
# This gives the best MAE
grid_search.best_score_

np.float64(-3.4005644051108304)

In [33]:
cv_results_df = pd.DataFrame(grid_search.cv_results_)

result_cols = [
    "param_learning_rate",
    "param_max_depth",
    "param_min_child_weight",
    "mean_test_mae",
    "std_test_mae",
    "mean_test_rmse",
    "std_test_rmse",
    "mean_test_mape",
    "std_test_mape",
    "mean_test_r2_score",
    "std_test_r2_score",
    "rank_test_mae",
    "rank_test_rmse",
    "rank_test_mape",
    "rank_test_r2_score"
]

cv_results_df[result_cols].iloc[grid_search.best_index_]

param_learning_rate       0.300000
param_max_depth           4.000000
param_min_child_weight    2.000000
mean_test_mae            -3.400564
std_test_mae              0.426908
mean_test_rmse           -5.276220
std_test_rmse             1.210392
mean_test_mape           -0.111880
std_test_mape             0.019255
mean_test_r2_score        0.922655
std_test_r2_score         0.037751
rank_test_mae             1.000000
rank_test_rmse            3.000000
rank_test_mape            6.000000
rank_test_r2_score        3.000000
Name: 81, dtype: float64

In [34]:
cv_results_df[result_cols][cv_results_df["rank_test_rmse"] == 1]

,param_learning_rate,param_max_depth,param_min_child_weight,mean_test_mae,std_test_mae,mean_test_rmse,std_test_rmse,mean_test_mape,std_test_mape,mean_test_r2_score,std_test_r2_score,rank_test_mae,rank_test_rmse,rank_test_mape,rank_test_r2_score
80,0.3,4,1,-3.421484,0.352055,-5.212572,1.083865,-0.110898,0.018811,0.924877,0.033916,2,1,3,1


In [35]:
cv_results_df[result_cols][cv_results_df["rank_test_r2_score"] == 1]

,param_learning_rate,param_max_depth,param_min_child_weight,mean_test_mae,std_test_mae,mean_test_rmse,std_test_rmse,mean_test_mape,std_test_mape,mean_test_r2_score,std_test_r2_score,rank_test_mae,rank_test_rmse,rank_test_mape,rank_test_r2_score
80,0.3,4,1,-3.421484,0.352055,-5.212572,1.083865,-0.110898,0.018811,0.924877,0.033916,2,1,3,1


In [36]:
mae = -cv_results_df["mean_test_mae"].iloc[grid_search.best_index_]
rmse = -cv_results_df["mean_test_rmse"].iloc[grid_search.best_index_]
mape = -cv_results_df["mean_test_mape"].iloc[grid_search.best_index_]
r2 = cv_results_df["mean_test_r2_score"].iloc[grid_search.best_index_]
n_samples, n_features = len(y_full_train), len(features)
r2_adj = 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)

results = [{
    "type": "xgboost",
    "mae_val": mae,
    "rmse_val": rmse,
    "mape_val": mape,
    "r2_val": r2,
    "adj_r2_val": r2_adj
}]

scores_df = pd.concat([scores_df, pd.DataFrame(results)], ignore_index=True)
scores_df.round(4)

,type,mae_val,rmse_val,mape_val,r2_val,adj_r2_val
0,random forest,4.2578,6.3809,0.1345,0.8906,0.8897
1,xgboost,3.4006,5.2762,0.1119,0.9227,0.9220


## 5 - Model development (Support Vector Machines)

- Because SVM is a distance-based model, it's a good practice to use a scaler.
- For finetuning, consider the `C` and `epsilon` parameter.

In [37]:
model = make_pipeline(
    StandardScaler(),
    SVR()
)
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('svr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['cement','blast_furnace_slag','fly_ash',...,'log_age','ratio_wb', 'pct_scm_repl']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [38]:
# Baseline score for SVM

y_pred_val = predict(
    model=model, 
    features_X=X_val, 
    target_y=y_val,
    set_type="validation"
)

===== Scores - VALIDATION =====
MAE: 7.1923
RMSE: 9.7857
MAPE: 0.2653
R2: 0.7153
R2 Adjusted: 0.7053


In [39]:
model = make_pipeline(
    StandardScaler(),
    SVR()
)

param_grid = {
    "svr__C": [0.1, 1, 10, 100, 1000],
    "svr__epsilon": [0.01, 0.1, 0.2, 0.5, 0.8, 1]
}

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "mape": "neg_mean_absolute_percentage_error",
    "r2_score": "r2"
}

grid_search = GridSearchCV(
    model, 
    param_grid=param_grid, 
    cv=cv_full_train, 
    scoring=scoring,
    refit="mae",
    verbose=2
)

grid_search.fit(X_full_train, y_full_train)

Fitting 10 folds for each of 30 candidates, totalling 300 fits
[CV] END ......................svr__C=0.1, svr__epsilon=0.01; total time=   0.0s
[CV] END ......................svr__C=0.1, svr__epsilon=0.01; total time=   0.0s
[CV] END ......................svr__C=0.1, svr__epsilon=0.01; total time=   0.0s
[CV] END ......................svr__C=0.1, svr__epsilon=0.01; total time=   0.0s
[CV] END ......................svr__C=0.1, svr__epsilon=0.01; total time=   0.0s
[CV] END ......................svr__C=0.1, svr__epsilon=0.01; total time=   0.0s
[CV] END ......................svr__C=0.1, svr__epsilon=0.01; total time=   0.0s
[CV] END ......................svr__C=0.1, svr__epsilon=0.01; total time=   0.0s
[CV] END ......................svr__C=0.1, svr__epsilon=0.01; total time=   0.0s
[CV] END ......................svr__C=0.1, svr__epsilon=0.01; total time=   0.0s
[CV] END .......................svr__C=0.1, svr__epsilon=0.1; total time=   0.0s
[CV] END .......................svr__C=0.1, sv

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...svr', SVR())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'svr__C': [0.1, 1, ...], 'svr__epsilon': [0.01, 0.1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'mae': 'neg_mean_absolute_error', 'mape': 'neg_mean_absolute_percentage_error', 'r2_score': 'r2', 'rmse': 'neg_root_mean_squared_error'}"
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'mae'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[(array([ 0, ...shape=(1170,)), ...), (array([ 0, ...shape=(1171,)), ...), ...]"
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed loggi

In [40]:
model_svr = grid_search.best_estimator_
model_svr

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('svr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['cement','blast_furnace_slag','fly_ash',...,'log_age','ratio_wb', 'pct_scm_repl']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [41]:
grid_search.best_params_

{'svr__C': 1000, 'svr__epsilon': 0.5}

In [42]:
# This gives the best MAE
grid_search.best_score_

np.float64(-4.015865855827206)

In [43]:
cv_results_df = pd.DataFrame(grid_search.cv_results_)

result_cols = [
    "param_svr__C",
    "param_svr__epsilon",
    "mean_test_mae",
    "std_test_mae",
    "mean_test_rmse",
    "std_test_rmse",
    "mean_test_mape",
    "std_test_mape",
    "mean_test_r2_score",
    "std_test_r2_score",
    "rank_test_mae",
    "rank_test_rmse",
    "rank_test_mape",
    "rank_test_r2_score"
]

cv_results_df[result_cols].iloc[grid_search.best_index_]

param_svr__C          1000.000000
param_svr__epsilon       0.500000
mean_test_mae           -4.015866
std_test_mae             0.334444
mean_test_rmse          -6.573857
std_test_rmse            0.907265
mean_test_mape          -0.117729
std_test_mape            0.018000
mean_test_r2_score       0.882479
std_test_r2_score        0.038292
rank_test_mae            1.000000
rank_test_rmse           3.000000
rank_test_mape           1.000000
rank_test_r2_score       2.000000
Name: 27, dtype: float64

In [44]:
mae = -cv_results_df["mean_test_mae"].iloc[grid_search.best_index_]
rmse = -cv_results_df["mean_test_rmse"].iloc[grid_search.best_index_]
mape = -cv_results_df["mean_test_mape"].iloc[grid_search.best_index_]
r2 = cv_results_df["mean_test_r2_score"].iloc[grid_search.best_index_]
n_samples, n_features = len(y_full_train), len(features)
r2_adj = 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)

results = [{
    "type": "svm",
    "mae_val": mae,
    "rmse_val": rmse,
    "mape_val": mape,
    "r2_val": r2,
    "adj_r2_val": r2_adj
}]

scores_df = pd.concat([scores_df, pd.DataFrame(results)], ignore_index=True)
scores_df.round(4)

,type,mae_val,rmse_val,mape_val,r2_val,adj_r2_val
0,random forest,4.2578,6.3809,0.1345,0.8906,0.8897
1,xgboost,3.4006,5.2762,0.1119,0.9227,0.9220
2,svm,4.0159,6.5739,0.1177,0.8825,0.8815


## 6 - Model summary

1. Random Forest
    - `n_estimators` = 300
    - `max_depth` = 25
    - `min_samples_leaf` = 1
2. XGBoost
    - `n_estimators` = 300
    - `learning_rate` = 0.3
    - `max_depth` = 4
    - `min_child_weight` = 2
3. Support Vector Machines
    - `C` = 1000
    - `epsilon` = 0.5

In [45]:
model_rfr

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",25
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",13
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of 

In [46]:
model_xgbr

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [47]:
model_svr

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('svr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['cement','blast_furnace_slag','fly_ash',...,'log_age','ratio_wb', 'pct_scm_repl']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [48]:
# Summary of scores
scores_df.round(4)

,type,mae_val,rmse_val,mape_val,r2_val,adj_r2_val
0,random forest,4.2578,6.3809,0.1345,0.8906,0.8897
1,xgboost,3.4006,5.2762,0.1119,0.9227,0.9220
2,svm,4.0159,6.5739,0.1177,0.8825,0.8815


Based on the scores, we choose **XGBoost** to as the final model.

In [49]:
# FI for XGBoost
importances_xgb = pd.Series(
    data=model_xgbr.feature_importances_,
    index=model_xgbr.feature_names_in_
)
importances_xgb.sort_values(ascending=False)

ratio_wb              0.476040
log_age               0.148613
cement                0.073777
coarse_aggregate      0.064163
pct_scm_repl          0.058336
silica_fume           0.043245
water                 0.038828
fly_ash               0.034110
fine_aggregate        0.022905
superplasticizer      0.022673
blast_furnace_slag    0.017311
dtype: float32

- By looking at the feature importance, we expect that the calculated ratio between the amount of water and the amount of binder (cement and SCMs) `ratio_wb` will significantly dictate the predicted strength.

## 7 - Final model

- Train the model using XGBoost with the best parameters then check test scores.
- Use both train and validation set to create the model.

In [50]:
features_X = df[features]
target_y = df[target]

X_full_train, X_test, y_full_train, y_test = train_test_split(
    features_X,
    target_y,
    test_size=0.2, 
    random_state=RANDOM_SEED
)

In [51]:
model_final = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=4,
    min_child_weight=2,
    learning_rate=0.3,
    random_state=RANDOM_SEED
)
model_final.fit(X_full_train, y_full_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [52]:
y_pred_test = predict(
    model=model_final, 
    features_X=X_test, 
    target_y=y_test,
    set_type="test"
)

===== Scores - TEST =====
MAE: 2.8282
RMSE: 4.2429
MAPE: 0.0894
R2: 0.9559
R2 Adjusted: 0.9544


This confirms the model generalizes well in unseen data.